In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
SOURCE_DIR = Path("src")
OUTPUT_DIR = Path("out")
OUTPUT_DIR.mkdir(exist_ok=True)

BY_ARTICLE_PATH = SOURCE_DIR / "by_article.csv"
YEAR_TOTALS_PATH = SOURCE_DIR / "year_totals.csv"

for path in [BY_ARTICLE_PATH, YEAR_TOTALS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")

In [ ]:
by_article = pd.read_csv(BY_ARTICLE_PATH)
year_totals = pd.read_csv(YEAR_TOTALS_PATH)

by_article["Wert_Anzahl"] = pd.to_numeric(by_article["Wert_Anzahl"], errors="coerce")
year_totals["Wert_Anzahl"] = pd.to_numeric(year_totals["Wert_Anzahl"], errors="coerce")

In [ ]:
year_summary = (
    year_totals.pivot_table(
        index="source_year",
        columns="Personengruppe",
        values="Wert_Anzahl",
        aggfunc="first",
    )
    .rename(
        columns={
            "Abgeurteilte insgesamt": "abgeurteilte",
            "Verurteilte insgesamt": "verurteilte",
        }
    )
    .reset_index()
    .sort_values("source_year")
)

year_summary["verurteilungsquote"] = year_summary["verurteilte"] / year_summary["abgeurteilte"]
year_summary["verurteilungsquote_pct"] = year_summary["verurteilungsquote"] * 100

year_summary

In [ ]:
article_base = by_article.loc[
    (by_article["Geschlecht"] == "Insgesamt")
    & (by_article["Altersgruppe"] == ".")
    & (by_article["Angewandtes_Strafrecht"] == ".")
    & (by_article["Personengruppe"].isin(["Abgeurteilte insgesamt", "Verurteilte insgesamt"]))
].copy()

article_summary = (
    article_base.pivot_table(
        index=["source_year", "Straftat_Gesetz", "Art_der_Straftat"],
        columns="Personengruppe",
        values="Wert_Anzahl",
        aggfunc="first",
    )
    .rename(
        columns={
            "Abgeurteilte insgesamt": "abgeurteilte",
            "Verurteilte insgesamt": "verurteilte",
        }
    )
    .reset_index()
)

article_summary = article_summary.dropna(subset=["abgeurteilte", "verurteilte"])
article_summary = article_summary.loc[article_summary["abgeurteilte"] > 0].copy()
article_summary["verurteilungsquote"] = article_summary["verurteilte"] / article_summary["abgeurteilte"]
article_summary["verurteilungsquote_pct"] = article_summary["verurteilungsquote"] * 100

article_summary.sort_values(["source_year", "abgeurteilte"], ascending=[True, False]).head(20)

In [ ]:
YEAR_SUMMARY_OUTPUT = OUTPUT_DIR / "year_conviction_rates.csv"
ARTICLE_SUMMARY_OUTPUT = OUTPUT_DIR / "article_conviction_rates.csv"

year_summary.to_csv(YEAR_SUMMARY_OUTPUT, index=False)
article_summary.to_csv(ARTICLE_SUMMARY_OUTPUT, index=False)

print(f"Wrote {YEAR_SUMMARY_OUTPUT} ({len(year_summary)} rows)")
print(f"Wrote {ARTICLE_SUMMARY_OUTPUT} ({len(article_summary)} rows)")